In [ ]:
# Software Name : mislabeled-benchmark
# SPDX-FileCopyrightText: Copyright (c) Orange Innovation
# SPDX-License-Identifier: MIT
#
# This software is distributed under the MIT license,
# see the "LICENSE.md" file for more details
# or https://github.com/Orange-OpenSource/mislabeled-benchmark/blob/master/LICENSE.md

In [ ]:
import os
import math
from define_models import baselines
import matplotlib.pyplot as plt
from IPython.display import display
# from critdd import Diagram, Diagrams
from collections import defaultdict
from numbers import Number
import texfig as tf
from texfig import TMLR_textwidth
import os
import pandas as pd
import numpy as np
import h5py
from benchmark_analysis import load_estim
from wilco_holm import draw_cd_diagram

In [ ]:
output_dir = "Projects/mislabeled-benchmark/ecai-output-sigmoid-2"
# output_dir = "Projects/mislabeled-benchmark/ecai-sigmoid-cifar-output"

In [ ]:
result_dirs = [
    (
        "weak/klm",
        os.path.join(os.path.expanduser("~"), f"{output_dir}/weak/klm"),
    ),
    # (
    #     "noise/klm",
    #     os.path.join(os.path.expanduser("~"), f"{output_dir}/noise/klm"),
    # ),
]


In [ ]:
def custom_grid(axis):
    axis.grid(c="#f2f2f2", which="both")
    axis.set_axisbelow(True)

In [ ]:
all_results = dict()

In [ ]:
all_results["allclass"] = dict()
for prefix, result_dir in result_dirs:
    try:
        all_results["allclass"][prefix] = load_estim(result_dir=result_dir)
    except:
        print(result_dir)

In [ ]:
detectors = np.unique(
    np.concatenate([r.detector_name.unique() for r in all_results['allclass'].values()])
)
datasets = np.unique(np.concatenate([r.dataset_name.unique() for r in all_results['allclass'].values()]))

detectors_nobaseline = list(set(detectors) - set(baselines + ["random"]))

len(detectors), detectors, len(detectors_nobaseline), detectors_nobaseline, len(
    datasets
), datasets

In [ ]:
d_base_model_map = dict()
d_detect_map = dict()

for d in detectors_nobaseline:
    s = d.split("_")
    if len(s) >= 2:
        d_base_model_map[d] = s[0]
    else:
        d_base_model_map[d] = "klm"

    d_detect_map[d] = " ".join(s[1:3])

detector_suffixes = np.unique(list(d_detect_map.values()))
detector_base = np.unique([v.split(" ")[0] for v in detector_suffixes])
d_base_model_map, d_detect_map, detector_suffixes, detector_base

In [ ]:
# detect_pretty_name = {
#     "agra": "AGRA",
#     "aum": "AUM",
#     "calibrated": "(calibrated)",
#     "isotonic": "(calibrated isotonic)",
#     "sigmoid": "(calibrated sigmoid)",
#     "noisy": "(calibrated noisy)",
#     "baseline": "(baseline)",
#     "adjusted": "(adjusted)",
#     "cleanlab": "Cleanlab",
#     "consensus": "Consensus",
#     "forget": "Forget scores",
#     "influence": "Influence",
#     "representer": "Representer",
#     "smallloss": "Small loss",
#     "tracin": "TracIn",
#     "vosg": "VoSG",
# }

detect_pretty_name = {
    "agra": "AGRA",
    "aum": "AUM",
    "isotonic": "calibrated (isotonic)",
    "sigmoid": "calibrated (sigmoid)",
    "noisy": "(calibrated noisy)",
    "baseline": "baseline",
    "adjusted": "adjusted",
    "cleanlab": "Cleanlab",
    "consensus": "Consensus",
    "forget": "Forget scores",
    "influence": "Influence",
    "representer": "Representer",
    "smallloss": "Small Loss",
    "tracin": "TracIn",
    "vosg": "VoSG",
}

bm_pretty_name = {"klm": "KLM", "gb": "GBT"}

In [ ]:
from matplotlib import colormaps
import seaborn as sns

# tab_colors = colormaps["tab20"]
# tab_colors = colormaps["tab10"]
tab_colors = sns.color_palette("Set2", len(detector_base))


def detector_color(detector_name):
    base = detector_name.split(" ")[0]
    mapping = {d: tab_colors[i] for i, d in enumerate(detector_base)}
    return mapping[base]


# detector_colors = {
#     d: tab_colors.colors[i % len(tab_colors.colors)]
#     for i, d in enumerate(detector_suffixes)
# }

plt.figure()
labels = []
# KLM
for d_name in detector_suffixes:
    l = f"({bm_pretty_name['klm']}) " + " ".join(
        [detect_pretty_name[d] for d in d_name.split(" ")[:2]]
    )
    plt.scatter([], [], label=l, color=detector_color(d_name))
    labels.append(l)

# GB
# for d_name, d_color in detector_colors.items():
#     if (
#         len(list(filter(lambda d: ("gb" in d) and (d_name in d), detectors_nobaseline)))
#         >= 1
#     ):
#         l = f"({bm_pretty_name['gb']}) " + " ".join(
#             [detect_pretty_name[d] for d in d_name.split(" ")]
#         )
#         plt.scatter([], [], label=l, color=d_color, marker="s")
#         labels.append(l)
#     else:
#         plt.scatter([], [], label="", color="white", alpha=0)
#         labels.append("")
for d_name in detector_suffixes:
    l = f"({bm_pretty_name['gb']}) " + " ".join(
        [detect_pretty_name[d] for d in d_name.split(" ")[:2]]
    )
    plt.scatter([], [], label=l, color=detector_color(d_name))
    labels.append(l)


disp_baselines = False
if disp_baselines:
    plt.scatter([], [], label="gold", color="black", marker="x")
    labels.append("gold")
    plt.scatter([], [], label="silver", color="black", marker="+")
    labels.append("silver")
    plt.scatter([], [], label="random", color="black", marker="p")
    labels.append("random")

    for i in range(len(detector_colors.items()) - 3):
        plt.scatter([], [], label="", color="white", alpha=0)
        labels.append("")

plt.gca().axis("off")
plt.legend(ncols=3 if disp_baselines else 2, labels=labels, title="Detectors")
# plt.savefig(f"figures/summary/{prefix_txt}_legend.pdf")

In [ ]:
full_detector_colors = dict()
full_detector_markers = dict()
for d in detectors:
    if d in baselines + ['random']:
        full_detector_colors[d] = 'white'
        full_detector_markers[d] = 'x'
    else:
        full_detector_colors[d] = detector_color(d_detect_map[d])
        full_detector_markers[d] = 'o' if d_base_model_map[d] == 'klm' else 's'


In [ ]:
for k, r in all_results['allclass'].items():
    d = r.pivot_table(
        index="detector_name",
        columns="dataset_name",
        values="estim_time",
        aggfunc="count",
    )
    print(k, d.sum().sum(), d.sum())
    display(d)

In [ ]:
for k, r in all_results['allclass'].items():
    print(k)
    display(
        r.pivot_table(
            index="detector_name",
            columns="dataset_name",
            values="logl_test",
            aggfunc="min",
        )
    )

In [ ]:
# Hyperparameters selection

all_results_cv = dict()
all_val_cv = dict()

baselines_norandom = set(baselines) - {"random"}
for which, _all_res in [("allclass", all_results["allclass"])]:
    all_results_cv[which] = dict()
    all_val_cv[which] = dict()

    for prefix in _all_res.keys():
        all_results_cv[which][prefix] = dict()
        all_val_cv[which][prefix] = dict()

        for cv_k, cv_function in [
            ("logl", lambda d, k: d[f"logl_{k}"].idxmin()),
            ("bacc", lambda d, k: d[f"bacc_{k}"].idxmax()),
            ("ece", lambda d, k: d[f"ece_{k}"].idxmin()),
        ]:
            indices_oracl_cv = cv_function(
                _all_res[prefix].groupby(["dataset_name", "detector_name"]), "test"
            )
            indices_clean_cv = cv_function(
                _all_res[prefix].groupby(["dataset_name", "detector_name"]), "val"
            )
            indices_noisy_cv = cv_function(
                _all_res[prefix].groupby(["dataset_name", "detector_name"]), "noisy_val"
            )
            indices_noisy_10_cv = cv_function(
                _all_res[prefix][
                    _all_res[prefix]["params_splitter"].apply(
                        lambda r: "quantile" in r.keys() and r["quantile"] == 0.1
                    )
                    | _all_res[prefix]["detector_name"].isin(baselines_norandom)
                ].groupby(["dataset_name", "detector_name"]),
                "noisy_val",
            )
            indices_noisy_90_cv = cv_function(
                _all_res[prefix][
                    _all_res[prefix]["params_splitter"].apply(
                        lambda r: "quantile" in r.keys() and r["quantile"] == 0.9
                    )
                    | _all_res[prefix]["detector_name"].isin(baselines_norandom)
                ].groupby(["dataset_name", "detector_name"]),
                "noisy_val",
            )
            indices_clean_90_cv = cv_function(
                _all_res[prefix][
                    _all_res[prefix]["params_splitter"].apply(
                        lambda r: "quantile" in r.keys() and r["quantile"] == 0.9
                    )
                    | _all_res[prefix]["detector_name"].isin(baselines_norandom)
                ].groupby(["dataset_name", "detector_name"]),
                "val",
            )
            indices_clean_10_cv = cv_function(
                _all_res[prefix][
                    _all_res[prefix]["params_splitter"].apply(
                        lambda r: "quantile" in r.keys() and r["quantile"] == 0.1
                    )
                    | _all_res[prefix]["detector_name"].isin(baselines_norandom)
                ].groupby(["dataset_name", "detector_name"]),
                "val",
            )

            all_results_cv[which][prefix][cv_k] = {
                "oracl": _all_res[prefix].loc[indices_oracl_cv.dropna()],
                "clean": _all_res[prefix].loc[indices_clean_cv.dropna()],
                "noisy": _all_res[prefix].loc[indices_noisy_cv.dropna()],
                "noisy_10": _all_res[prefix].loc[indices_noisy_10_cv.dropna()],
                "noisy_90": _all_res[prefix].loc[indices_noisy_90_cv.dropna()],
                "clean_90": _all_res[prefix].loc[indices_clean_90_cv.dropna()],
                "clean_10": _all_res[prefix].loc[indices_clean_10_cv.dropna()],
            }

            all_val_cv[which][prefix][cv_k] = {
                k: r.pivot(
                    index="detector_name", columns="dataset_name", values=f"{cv_k}_test"
                )
                for k, r in all_results_cv[which][prefix][cv_k].items()
            }

        # sanity check
        best_logl_direct = _all_res[prefix].pivot_table(
            index="detector_name",
            columns="dataset_name",
            values="logl_test",
            aggfunc="min",
        )

        assert (
            best_logl_direct - all_val_cv[which][prefix]["logl"]["oracl"]
        ).abs().sum().sum() == 0

        best_bacc_direct = _all_res[prefix].pivot_table(
            index="detector_name",
            columns="dataset_name",
            values="bacc_test",
            aggfunc="max",
        )

        assert (
            best_bacc_direct - all_val_cv[which][prefix]["bacc"]["oracl"]
        ).abs().sum().sum() == 0

hp_tune_str = {
    "oracl": "oracle",
    "clean": "clean validation set",
    "noisy": "noisy validation set",
    "noisy_10": "noisy validation set (threshold=10%)",
}

In [ ]:
prefix = "weak/klm"
exp_alt = "allclass"


results_cv = all_results_cv[exp_alt][prefix]["logl"]
logl_cv = all_val_cv[exp_alt][prefix]["logl"]
logl_cv_for_norm = all_val_cv["allclass"][prefix]["logl"]

roc_auc = results_cv["oracl"].pivot(
    index="detector_name", columns="dataset_name", values="global_ranking_quality"
)

n_per_row = 4
n_rows = math.ceil(len(datasets) / n_per_row)
fig, axes = plt.subplots(n_rows, n_per_row, figsize=(13, 3 * n_rows))

for i, dataset_name in enumerate(results_cv["oracl"].dataset_name.unique()):
    axis = axes[i // n_per_row, i % n_per_row]

    axis.set_title(dataset_name)

    d_colors = [full_detector_colors[d] for d in roc_auc.index]

    axis.scatter(
        roc_auc[dataset_name],
        logl_cv["oracl"][dataset_name],
        marker="+",
        c=d_colors,
    )

    perf_none = logl_cv_for_norm["oracl"][dataset_name]["none"]
    perf_silver = logl_cv_for_norm["oracl"][dataset_name]["silver"]
    axis.hlines(perf_none, 0.35, 1)
    axis.hlines(perf_silver, 0.35, 1)

    # axis.set_xlim(0.35, 1)
    print(perf_none, perf_silver)
    axis.set_ylim(2 * perf_silver - perf_none, 2 * perf_none - perf_silver)
    custom_grid(axis)

for axis in axes:
    axis[0].set_ylabel("test log loss")
for axis in axes[-1]:
    axis.set_xlabel("roc auc")

plt.tight_layout()
plt.show()

In [ ]:
prefix = "weak/klm"
exp_alt = "allclass"

results_cv = all_results_cv[exp_alt][prefix]["logl"]
logl_cv = all_val_cv[exp_alt][prefix]["logl"]
logl_cv_for_norm = all_val_cv["allclass"][prefix]["logl"]

n_per_row = 4
n_rows = math.ceil(len(datasets) / n_per_row)
fig, axes = plt.subplots(n_rows, n_per_row, figsize=(13, 3 * n_rows))

for i, dataset_name in enumerate(results_cv["oracl"].dataset_name.unique()):
    axis = axes[i // n_per_row, i % n_per_row]

    axis.set_title(dataset_name)

    sorted_indices = np.argsort(np.argsort(logl_cv["oracl"][dataset_name].values))
    n_detectors = len(sorted_indices)

    d_colors = [full_detector_colors[d] for d in logl_cv["oracl"][dataset_name].index]

    perf_none = logl_cv_for_norm["oracl"][dataset_name]["none"]
    perf_silver = logl_cv_for_norm["oracl"][dataset_name]["silver"]
    axis.hlines(perf_none, -1, n_detectors, alpha=0.6)
    axis.hlines(perf_silver, -1, n_detectors)

    axis.scatter(
        sorted_indices,
        logl_cv["oracl"][dataset_name],
        marker="_",
        c=d_colors,
    )
    axis.scatter(
        sorted_indices,
        logl_cv["clean"][dataset_name],
        marker="x",
        c=d_colors,
    )

    axis.scatter(
        sorted_indices,
        logl_cv["noisy"][dataset_name],
        marker="*",
        c=d_colors,
    )

    # axis.set_ylim(2 * perf_silver - perf_none, 2 * perf_none - perf_silver)
    axis.set_xlim(-1, n_detectors)
    custom_grid(axis)

for axis in axes:
    axis[0].set_ylabel("test log loss")

if not os.path.exists("figures/summary"):
    os.mkdir("figures/summary/")

plt.savefig(f"figures/summary/{prefix.replace('/', '_')}.pdf", bbox_inches="tight")
plt.show()

In [ ]:
results_cv.keys()

In [ ]:
hp_tune_str = {
    "clean": "HP tuned on noise free validation",
    "noisy": "HP tuned on noisy validation",
    "oracl": "Oracle HP",
    "noisy_10": "HP tuned on noisy validation, threshold forced at 10\%",
    "noisy_90": "HP tuned on noisy validation, training on 10\% top trusted",
    "clean_90": "HP tuned on clean validation, training on 10\% top trusted",
    "clean_10": "HP tuned on clean validation, training on 90\% top trusted",
}

strategy_str = {
    "allclass": "Filtering",
    "byclass": "Filtering by class",
    "relabel": "10\% relabeling",
}

In [ ]:
for cal in ["isotonic", "sigmoid"]:
    prout = pd.melt(
        all_val_cv["allclass"]["weak/klm"]["logl"]["clean"].reset_index(),
        id_vars="detector_name",
    ).rename(
        {"detector_name": "competitor", "dataset_name": "dataset", "value": "metric"},
        axis=1,
    )
    prout["calibration_size"] = np.nan
    prout.loc[prout["competitor"].str.contains(r"[0-9]"), "calibration_size"] = (
        prout.loc[prout["competitor"].str.contains(r"[0-9]"), "competitor"]
        .str.split("_")
        .str[-1]
        .astype(float)
    )
    prout_filtered = prout[
        (
            (
                (prout["calibration_size"] == 1.0)
                .__and__(~prout["competitor"].str.contains("noisy"))
                .__and__(prout["competitor"].str.contains(cal))
            ).__or__(
                prout["calibration_size"]
                .isna()
                .__and__(~prout["competitor"].str.contains("adjusted"))
                .__and__(~prout["competitor"].str.contains("isotonic"))
                .__and__(~prout["competitor"].str.contains("sigmoid"))
            )
        )
    ]
    prout_filtered["competitor"] = prout_filtered["competitor"].apply(
        lambda name: (
            detect_pretty_name[name.split("_")[1]] + " (" + cal[:3] + ".)"
            if len(name.split("_")) > 2
            else detect_pretty_name[name.split("_")[1]]
        )
        if len(name.split("_")) > 1
        else name
    )

    os.makedirs(f"figures/critical/{cal}", exist_ok=True)

    print(prout_filtered)

    draw_cd_diagram(
        prout_filtered,
        ascending=True,
        pairwise_matrix="uncorrected",
        mode="nemenyi",
        save=f"figures/critical/{cal}",
    )


In [ ]:
from operator import lt, gt
from wilco_holm import wilcoxon_holm

for name, criteria, op, comp in [
    ("loss", "logl", "min", lt),
    ("balanced accuracy", "bacc", "max", gt),
]:
    proutprout = all_val_cv["allclass"]["weak/klm"][criteria]["clean"].transpose()
    proutprout = proutprout[
        [
            c
            for c in proutprout.columns
            if ("1.0" in c and "noisy" not in c)
            or (len(c.split("_")) <= 3)
            and c != "sigmoid"
            and c != "isotonic"
        ]
    ]
    # proutprout = proutprout.sub(proutprout["none"], axis=0).div(
    #     proutprout["silver"] - proutprout["none"] + 1e-10, axis=0
    # )
    # proutprout = (2 - proutprout) * 100
    # proutprout = proutprout.drop(["gold", "silver", "random", "none"], axis=1)
    proutprout = proutprout.drop(["gold", "random"], axis=1)
    proutprout = proutprout[
        ["silver"]
        + [c for c in proutprout.columns if c not in ["silver", "none"]]
        + ["none"]
    ]

    lastrow = []
    for c in proutprout.columns:
        c_base = "_".join(c.split("_")[:2])
        if c in baselines:
            lastrow.append("-")
        elif c == c_base:
            lastrow.append("-")
        else:
            wins = comp(proutprout[c], proutprout["_".join(c.split("_")[:2])]).sum()
            draws = (proutprout[c] == proutprout["_".join(c.split("_")[:2])]).sum()
            losses = len(proutprout[c]) - wins - draws
            pivoted = pd.melt(proutprout[[c, c_base]], ignore_index=False)
            pivoted = pivoted.reset_index()
            pivoted = pivoted.rename(
                {
                    "detector_name": "competitor",
                    "value": "metric",
                    "dataset_name": "dataset",
                },
                axis=1,
            )
            wdl = f"{wins}/{draws}/{losses}"
            if wilcoxon_holm(0.05, pivoted)[0][0][-1]:
                lastrow.append("\\underline{" + wdl + "}")
            else:
                lastrow.append(wdl)

    lastrow = (
        "wins/draws/losses & "
        + " & ".join([e if "-" in e else f"\\bfseries {e}" for e in lastrow])
        + r" \\"
    )

    proutprout = proutprout.rename(
        lambda c: (
            detect_pretty_name[c.split("_")[1]],
            "\\textit{" + c.split("_")[2][:3] + ".}",
        )
        if "1.0" in c
        else (
            (detect_pretty_name[c.split("_")[1]], "\\textit{base.}")
            if len(c.split("_")) == 2
            else (
                (detect_pretty_name[c.split("_")[1]], "\\textit{adj.}")
                if len(c.split("_")) == 3
                else tuple(["\\multirow{2.5}{*}{" + c + "}", "torm"])
            )
        ),
        axis=1,
    )
    proutprout.columns = pd.MultiIndex.from_tuples(
        proutprout.columns, names=["\\multirow{2.5}{*}{\\textbf{Dataset}}", "torm"]
    )

    proutprout.index = [i.replace("_", "-") for i in proutprout.index]

    proutprout = proutprout.round(2)
    with open(os.path.join("figures", f"table-results-{criteria}.tex"), "w") as table:
        latex = proutprout.style.format(precision=2)
        if op == "min":
            latex = (
                latex.highlight_min(
                    axis=1,
                    subset=["AUM"],
                    props="font-weight:bold",
                )
                .highlight_min(
                    axis=1,
                    subset=["Cleanlab"],
                    props="font-weight:bold",
                )
                .highlight_min(
                    axis=1,
                    subset=["Consensus"],
                    props="font-weight:bold",
                )
                .highlight_min(
                    axis=1,
                    subset=["Small Loss"],
                    props="font-weight:bold",
                )
            )
        else:
            latex = (
                latex.highlight_max(
                    axis=1,
                    subset=["AUM"],
                    props="font-weight:bold",
                )
                .highlight_max(
                    axis=1,
                    subset=["Cleanlab"],
                    props="font-weight:bold",
                )
                .highlight_max(
                    axis=1,
                    subset=["Consensus"],
                    props="font-weight:bold",
                )
                .highlight_max(
                    axis=1,
                    subset=["Small Loss"],
                    props="font-weight:bold",
                )
            )
        latex = latex.highlight_min(
            axis=1,
            subset=["\\multirow{2.5}{*}{silver}"],
            props="font-style:italic",
        ).highlight_min(
            axis=1,
            subset=["\\multirow{2.5}{*}{none}"],
            props="font-style:italic",
        )
        latex = latex.to_latex(
            hrules=True,
            multirow_align="t",
            multicol_align="c",
            position="h",
            label=f"tab:results-{criteria}",
            convert_css=True,
        )
        latex = latex.replace("torm", "")
        latex = latex.split("\n")
        latex[0] = latex[0].replace("}", "*}")
        latex.insert(1, "\\centering")
        latex.insert(
            2,
            "\\topcaption{"
            + "Test "
            + name
            + " of the best pipelines for each detector and add-on for all datasets. The results in \\textbf{bold} are the results obtained by the best add-on for each detector on each dataset. The results of the silver and none references are in \\textit{italic}."
            + "}",
        )
        latex.insert(4, "\\resizebox{\\linewidth}{!}{")
        latex.insert(
            8,
            "\\cmidrule(lr){3-6} \\cmidrule(lr){7-10} \\cmidrule(lr){11-14} \\cmidrule(lr){15-18}",
        )
        latex.insert(-2, "}")
        latex.insert(-5, "\\midrule")
        latex.insert(-5, lastrow)
        latex[-2] = latex[-2].replace("}", "*}")
        # for i, line in enumerate(latex):
        #     if line == "\cline{1-12} \cline{2-12}":
        #         latex[i] = "\cline{1-12}"
        table.write("\n".join(latex))

In [ ]:
prefix = "weak/klm"
exp_alt = "allclass"
metric = "logl"
default_calibration_size = str(1.0)

results_cv = all_results_cv[exp_alt][prefix][metric]
logl_cv = all_val_cv[exp_alt][prefix][metric]
logl_cv_for_norm = all_val_cv["allclass"][prefix][metric]

import seaborn as sns
import itertools
from matplotlib.patches import Patch

for k, r in results_cv.items():
    if exp_alt == "relabel" and k in ["noisy_10", "noisy_90", "clean_10", "clean_90"]:
        continue

    logl_cv_none = logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "none"].values
    # logl_cv_random = logl_cv[k].loc[logl_cv[k].index == "random"].values
    # logl_cv_wood = logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "wood"].values
    logl_cv_silver = (
        logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "silver"].values
    )
    # logl_cv_gold = logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "gold"].values

    # norm_low = (logl_cv_none + logl_cv_wood) / 2
    # norm_high = (logl_cv_silver + logl_cv_gold) / 2

    if False and exp_alt == "relabel":
        norm_low = logl_cv_random
    else:
        norm_low = logl_cv_none
    norm_high = logl_cv_silver

    print(k, norm_low.shape, norm_high.shape, logl_cv[k].shape)
    logl_cv_norm = (logl_cv[k] - norm_low) / (norm_high - norm_low + 1e-10)
    logl_cv_norm = (2 - logl_cv_norm) * 100

    df_sns_format = pd.melt(
        logl_cv_norm.reset_index(),
        id_vars="detector_name",
        value_vars=logl_cv_norm.columns,
    )
    na_mask = df_sns_format["detector_name"].isin(
        [b for b in baselines if b != "isotonic" and b != "sigmoid"] + ["random"]
    )
    calibrated_mask = (
        df_sns_format["detector_name"]
        .str.contains(default_calibration_size)
        .__and__(~df_sns_format["detector_name"].str.contains("noisy"))
    )
    adjusted_mask = df_sns_format["detector_name"].str.contains("adjust")

    selected_mask = calibrated_mask.__or__(
        ~df_sns_format["detector_name"].str.contains("isotonic|sigmoid")
    ).__and__(~na_mask)

    df_sns_format.loc[:, "addon"] = "baseline"
    df_sns_format.loc[calibrated_mask, "addon"] = (
        df_sns_format.loc[calibrated_mask, "detector_name"].str.split("_").str[2]
    )
    df_sns_format.loc[adjusted_mask, "addon"] = (
        df_sns_format.loc[adjusted_mask, "detector_name"].str.split("_").str[2]
    )
    df_sns_format.loc[na_mask, "addon"] = df_sns_format.loc[na_mask, "detector_name"]
    df_sns_format.loc[na_mask, "detector_name"] = "none"
    df_sns_format.loc[~na_mask, "detector_name"] = (
        df_sns_format.loc[~na_mask, "detector_name"]
        .str.split("_")
        .str[:2]
        .str.join("_")
    )

    ordered_detectors = sorted(
        df_sns_format.loc[selected_mask, "detector_name"].unique()
    )
    print(ordered_detectors)
    ordered_addons = [
        "none",
        "random",
        "baseline",
        "adjusted",
        "sigmoid",
        "isotonic",
        "silver",
        "gold",
    ]

    tf.figure(width=TMLR_textwidth * 0.95, ratio=0.5, pad=0.5)

    sns.boxplot(
        df_sns_format[selected_mask],
        x="addon",
        y="value",
        hue="detector_name",
        hue_order=ordered_detectors,
        palette=sns.color_palette([full_detector_colors[d] for d in ordered_detectors]),
        order=ordered_addons,
        showfliers=False,
        **{
            "boxprops": {"edgecolor": "black"},
            "medianprops": {"color": "black"},
            "whiskerprops": {"color": "black"},
            "capprops": {"color": "black"},
        },
        legend=False,
    )

    sns.boxplot(
        df_sns_format[na_mask],
        x="addon",
        y="value",
        order=ordered_addons,
        showfliers=False,
        width=0.2,
        **{
            "boxprops": {"edgecolor": "black", "facecolor": "white"},
            "medianprops": {"color": "black"},
            "whiskerprops": {"color": "black"},
            "capprops": {"color": "black"},
        },
        legend=False,
    )

    sns.stripplot(
        df_sns_format[selected_mask],
        x="addon",
        y="value",
        hue="detector_name",
        hue_order=ordered_detectors,
        palette=sns.color_palette([full_detector_colors[d] for d in ordered_detectors]),
        order=ordered_addons,
        size=3,
        dodge=True,
        edgecolor="black",
        linewidth=1,
        legend=False,
    )

    sns.stripplot(
        df_sns_format[na_mask],
        x="addon",
        y="value",
        size=3,
        dodge=True,
        edgecolor="black",
        facecolor="white",
        linewidth=1,
        legend=False,
    )

    pretty_xticks = []
    for s in ordered_addons:
        if s in baselines + ["random"]:
            if s == "calibrated":
                pretty_xticks.append("none-cal")
            else:
                if s in ["isotonic", "sigmoid"]:
                    pretty_xticks.append("calibrated\n(" + s.replace("_", "\\_") + ")")
                else:
                    pretty_xticks.append(s.replace("_", "\\_"))
        else:
            pretty_xticks.append("\n".join(detect_pretty_name[s].split(" ")))

    plt.xticks(
        range(len(ordered_addons)),
        pretty_xticks,
        style="italic",
    )

    # for i, detector_name in enumerate(ordered_detectors):
    #     if detector_name in baselines:
    #         plt.gca().get_xticklabels()[i].set_color("red")
    #     elif detector_name in ["random"]:
    #         plt.gca().get_xticklabels()[i].set_color("blue")

    plt.yticks([0, 300], [0, 300], minor=True)
    plt.yticks([100, 200], minor=False)

    plt.ylim(0, 300)
    custom_grid(plt.gca())
    plt.grid(axis="x")
    plt.grid(axis="y", which="major", linewidth=1, color="black", linestyle="dashed")
    plt.grid(axis="y", which="minor", linewidth=0, visible=True)

    # # plt.title(f"Hyperparameters tuned using {hp_tune_str[k]} | {prefix} | {exp_alt}")
    plt.ylabel("normalized test loss")
    plt.xlabel("")

    plt.legend(
        handles=[
            Patch(
                facecolor=sns.color_palette(
                    [full_detector_colors[d] for d in ordered_detectors]
                )[i],
                edgecolor="black",
                label=f"{detect_pretty_name[c.split('_')[-1]]}",
            )
            for i, c in enumerate(ordered_detectors)
        ],
        loc="upper right",
        # ncol=2,
        fontsize=9,
        title="Detector",
        # title_fontsize=9,
        title_fontproperties={"size": 9},
        edgecolor="black",
    )

    if not os.path.exists("figures/detectors"):
        # os.mkdir("figures")
        os.mkdir("figures/detectors/")

    tf.savefig(f"figures/detectors/{prefix.replace('/', '_')}_{exp_alt}_{k}_{metric}")

    plt.show()

In [ ]:
prefix = "weak/klm"
exp_alt = "allclass"
metric = "logl"
default_calibration_size = str(1.0)

results_cv = all_results_cv[exp_alt][prefix][metric]
logl_cv = all_val_cv[exp_alt][prefix][metric]
logl_cv_for_norm = all_val_cv["allclass"][prefix][metric]

import seaborn as sns
import itertools
from matplotlib.patches import Patch

for k, r in results_cv.items():
    if exp_alt == "relabel" and k in ["noisy_10", "noisy_90", "clean_10", "clean_90"]:
        continue

    logl_cv_none = logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "none"].values
    # logl_cv_random = logl_cv[k].loc[logl_cv[k].index == "random"].values
    # logl_cv_wood = logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "wood"].values
    logl_cv_silver = (
        logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "silver"].values
    )
    # logl_cv_gold = logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "gold"].values

    # norm_low = (logl_cv_none + logl_cv_wood) / 2
    # norm_high = (logl_cv_silver + logl_cv_gold) / 2

    if False and exp_alt == "relabel":
        norm_low = logl_cv_random
    else:
        norm_low = logl_cv_none
    norm_high = logl_cv_silver

    print(k, norm_low.shape, norm_high.shape, logl_cv[k].shape)
    logl_cv_norm = (logl_cv[k] - norm_low) / (norm_high - norm_low + 1e-10)
    logl_cv_norm = (2 - logl_cv_norm) * 100

    df_sns_format = pd.melt(
        logl_cv_norm.reset_index(),
        id_vars="detector_name",
        value_vars=logl_cv_norm.columns,
    )
    non_baseline_mask = ~df_sns_format["detector_name"].isin(
        ["none", "silver", "gold", "random", "isotonic", "sigmoid"]
    ).__or__(~df_sns_format["detector_name"].str.contains("isotonic")).__or__(
        df_sns_format["detector_name"].str.contains("noisy")
    )
    df_sns_format.loc[non_baseline_mask, "calibration_size"] = (
        df_sns_format.loc[non_baseline_mask, "detector_name"].str.split("_").str[-1]
    ).astype(float)
    non_baseline_mask = non_baseline_mask.__and__(df_sns_format["calibration_size"] > 1)
    df_sns_format.loc[non_baseline_mask, "detector_name"] = (
        df_sns_format.loc[non_baseline_mask, "detector_name"]
        .str.split("_")
        .str[:-1]
        .str.join("_")
    )

    ordered_detectors = sorted(
        df_sns_format.loc[non_baseline_mask, "detector_name"].unique()
    )
    ordered_addons = sorted(
        df_sns_format.loc[non_baseline_mask, "calibration_size"].unique()
    )

    tf.figure(width=TMLR_textwidth * 0.95, ratio=0.4, pad=0.5)

    # plt.axhline(100)
    # plt.axhline(200)

    ax = sns.boxplot(
        df_sns_format[non_baseline_mask],
        x="detector_name",
        y="value",
        hue="calibration_size",
        # hue_order=df_sns_format.loc[non_baseline_mask, "calibration_size"].sort_values(),
        order=ordered_detectors,
        hue_order=ordered_addons,
        # width=0.4,
        showfliers=False,
        **{
            "boxprops": {"edgecolor": "black"},
            "medianprops": {"color": "black"},
            "whiskerprops": {"color": "black"},
            "capprops": {"color": "black"},
        },
        legend=False,
        # palette=sum(
        #     [
        #         sns.light_palette(
        #             full_detector_colors[d + "_0.1"],
        #             n_colors=len(ordered_calibration_sizes),
        #         )
        #         for d in ordered_detectors
        #     ],
        #     [],
        # ),
        # palette = {d: full_detector_colors[d+"_0.1"] for d in df_sns_format.loc[non_baseline_mask, "detector_name"].unique()}
        # boxprops={}
    )

    print(ordered_addons, ordered_detectors)

    for i, box in enumerate(ax.patches):
        if i >= len(ordered_addons) * len(ordered_detectors):
            break
        i, j = i % len(ordered_detectors), i // len(ordered_detectors)
        palette = sns.light_palette(
            full_detector_colors[ordered_detectors[i] + "_" + default_calibration_size],
            n_colors=len(ordered_addons),
        )
        box.set_facecolor(palette[j])

    ax2 = sns.stripplot(
        df_sns_format[non_baseline_mask],
        x="detector_name",
        y="value",
        hue="calibration_size",
        dodge=True,
        size=3,
        hue_order=ordered_addons,
        order=ordered_detectors,
        # ax=ax,
        edgecolor="black",
        linewidth=1,
        legend=False,
        ax=ax,
    )

    for i, box in enumerate(ax2.collections):
        if i >= len(ordered_addons) * len(ordered_detectors):
            break
        i, j = i % len(ordered_addons), i // len(ordered_addons)
        palette = sns.light_palette(
            full_detector_colors[ordered_detectors[j] + "_" + default_calibration_size],
            n_colors=len(ordered_addons),
        )
        box.set_facecolor(palette[i])

    # for i, detector_name in enumerate(ordered_detectors):
    #     d = logl_cv_norm.loc[logl_cv_norm.index == detector_name].values[0]
    #     # print(len(d))

    #     d_base100 = (2 - d) * 100
    #     d_base100 = d_base100[~np.isnan(d_base100)]
    #     bplot = plt.boxplot(
    #         [d_base100],
    #         positions=[i],
    #         # notch=True,
    #         # bootstrap=100,
    #         widths=[0.7],
    #         showfliers=False,
    #         patch_artist=True,
    #     )

    #     if np.any(np.isnan(d_base100)):
    #         ccc

    #     bplot["boxes"][0].set_facecolor(full_detector_colors[detector_name])
    #     bplot["boxes"][0].set_alpha(0.8)

    #     eps = 0.05
    #     plt.scatter(
    #         [i] * len(d_base100) + np.random.uniform(-eps, eps, size=len(d_base100)),
    #         d_base100,
    #         facecolors=[full_detector_colors[detector_name]] * len(d_base100),
    #         edgecolors="black",
    #         s=[5] * len(d_base100),
    #         zorder=10,
    #     )

    pretty_xticks = []
    for s in ordered_detectors:
        if s in baselines + ["random"]:
            pretty_xticks.append(s.replace("_", "\\_"))
        else:
            pretty_xticks.append(
                " ".join([detect_pretty_name[d] for d in s.split("_")[1:-1]])
            )
    plt.xticks(
        range(len(ordered_detectors)),
        pretty_xticks,
        # rotation=50,
        # rotation_mode="anchor",
        # ha="right",
    )

    # for i, detector_name in enumerate(ordered_detectors):
    #     if detector_name in baselines:
    #         plt.gca().get_xticklabels()[i].set_color("red")
    #     elif detector_name in ["random"]:
    #         plt.gca().get_xticklabels()[i].set_color("blue")

    plt.yticks([0, 300], [0, 300], minor=True)
    plt.yticks([100, 200], minor=False)

    plt.ylim(0, 300)
    custom_grid(plt.gca())
    plt.grid(axis="x")
    plt.grid(axis="y", which="major", linewidth=1, color="black", linestyle="dashed")
    plt.grid(axis="y", which="minor", linewidth=0, visible=True)

    # # plt.title(f"Hyperparameters tuned using {hp_tune_str[k]} | {prefix} | {exp_alt}")
    plt.ylabel("normalized test loss")
    plt.xlabel("")

    plt.legend(
        handles=[
            Patch(
                facecolor=sns.light_palette("grey", n_colors=len(ordered_addons))[i],
                edgecolor="black",
                label=f"{int(c)}",
                # label=f"{int(100 * float(c))}%",
            )
            for i, c in enumerate(ordered_addons)
        ],
        loc="upper right",
        ncol=3,
        fontsize=9,
        title="calibration set size",
        # title="% of val. use for calib.",
        title_fontsize=9,
        edgecolor="black",
        columnspacing=0.8,
    )

    # noise_str = "NCAR" if prefix.split("/")[0] == "noise" else "NNAR"
    # classif_str = (
    #     "Linear Classifier"
    #     if prefix.split("/")[1] == "klm"
    #     else "Gradient Boosting Classifier"
    # )
    # # plt.suptitle(f"{hp_tune_str[k]}")
    # # plt.suptitle(
    # #     f"{noise_str} | {strategy_str[exp_alt]} | {classif_str} | {hp_tune_str[k]}"
    # # )

    if not os.path.exists("figures/calib_size"):
        # os.mkdir("figures")
        os.mkdir("figures/calib_size/")

    tf.savefig(f"figures/calib_size/{prefix.replace('/', '_')}_{exp_alt}_{k}_{metric}")

    plt.show()

In [ ]:
default_calibration_size = str(1.0)

def plot_perf_clean_vs_noisy_calib(prefix, k, exp_alt):
    logl_cv_weak = all_val_cv[exp_alt][prefix]["logl"][k]

    tf.figure(width=TMLR_textwidth * (2 / 5), ratio=1, pad=0.5)

    for detector in set(detectors) - set(baselines) - set(["random"]):

        if (
            "adjusted" in detector
            or "isotonic" in detector
            or "sigmoid" in detector
            or "nois" in detector
            or "gb" in detector
            or "forget" in detector
        ):
            continue
        print(detector)

        perf_noise = logl_cv_weak.loc[logl_cv_weak.index == f"{detector}_isotonic_noisy_{default_calibration_size}"]
        perf_clean = logl_cv_weak.loc[logl_cv_weak.index == f"{detector}_isotonic_{default_calibration_size}"]

        if perf_noise.shape != perf_clean.shape:
            return None

        plt.scatter(
            perf_noise,
            perf_clean,
            color=full_detector_colors[detector],
            s=12,
            alpha=0.7,
            # marker=full_detector_markers[detector],
        )

    xlims, ylims = plt.xlim(), plt.ylim()
    xy_min = min(xlims[0], ylims[0])
    xy_max = max(xlims[1], ylims[1])

    plt.plot([xy_min, xy_max], [xy_min, xy_max], color="black")
    plt.xlabel("test log loss\n(noisy calibration set)")
    plt.ylabel("test log loss\n(clean calibration set)")

    custom_grid(plt.gca())
    plt.xlim(1e-1, 1)
    plt.ylim(1e-1, 1)
    plt.xscale("log")
    plt.yscale("log")

    plt.xticks(np.arange(0.1, 1, 0.1), [], minor=True)
    plt.yticks(np.arange(0.1, 1, 0.1), [], minor=True)
    plt.xticks([0.1, 1], [0.1, 1], minor=False)
    plt.yticks([0.1, 1], [0.1, 1], minor=False)
    # plt.grid()

    tf.savefig(f"figures/perf_clean_vs_noisy_calib/{prefix.replace('/','_')}_{exp_alt}_{k}")
    plt.show()

if not os.path.exists("figures/perf_clean_vs_noisy_calib"):
    os.mkdir("figures/perf_clean_vs_noisy_calib")

plot_perf_clean_vs_noisy_calib(prefix="weak/klm", k="clean", exp_alt="allclass")
plot_perf_clean_vs_noisy_calib(prefix="weak/klm", k="oracl", exp_alt="allclass")

In [ ]:
import matplotlib.colors as mcolors
default_calibration_size = str(1.0)


def plot_perf_clean_vs_noisy_calib(prefix, k, exp_alt):
    logl_cv_weak = all_val_cv[exp_alt][prefix]["logl"][k]

    tf.figure(width=TMLR_textwidth * (2 / 5), ratio=1, pad=0.5)

    top = {"clean": 0, "noisy": 0}
    bottom = {"clean": 0, "noisy": 0}
    for detector in set(detectors) - set(baselines) - set(["random"]):
        if (
            "adjusted" in detector
            or "isotonic" in detector
            or "sigmoid" in detector
            or "nois" in detector
            or "gb" in detector
            or "forget" in detector
        ):
            continue

        print(detector)

        perf_nothing = logl_cv_weak.loc[logl_cv_weak.index == f"{detector}"]
        perf_noise = logl_cv_weak.loc[
            logl_cv_weak.index
            == f"{detector}_isotonic_noisy_{default_calibration_size}"
        ]
        perf_clean = logl_cv_weak.loc[
            logl_cv_weak.index == f"{detector}_isotonic_{default_calibration_size}"
        ]

        if perf_noise.shape != perf_clean.shape:
            return None

        plt.scatter(
            perf_nothing,
            perf_clean,
            color=mcolors.XKCD_COLORS["xkcd:neon pink"],
            s=12,
            alpha=0.7,
        )
        plt.scatter(
            perf_nothing,
            perf_noise,
            color=mcolors.XKCD_COLORS["xkcd:neon blue"],
            s=12,
            alpha=0.7,
        )
        top["clean"] += (perf_clean.values > perf_nothing.values).sum()
        top["noisy"] += (perf_noise.values > perf_nothing.values).sum()
        bottom["clean"] += (perf_clean.values < perf_nothing.values).sum()
        bottom["noisy"] += (perf_noise.values < perf_nothing.values).sum()

    print(top, bottom)

    xlims, ylims = plt.xlim(), plt.ylim()
    xy_min = min(xlims[0], ylims[0])
    xy_max = max(xlims[1], ylims[1])

    plt.plot([0.1, 2], [0.1, 2], color="black")

    # plt.plot(
    #     np.linspace(0.1, 2, 100, endpoint=True),
    #     np.linspace(0.1, 2, 100, endpoint=True) - 0.1,
    #     color="black",
    #     linestyle="dashed",
    # )
    # plt.plot(
    #     np.linspace(0.1, 2, 100, endpoint=True),
    #     np.linspace(0.1, 2, 100, endpoint=True) + 0.1,
    #     color="black",
    #     linestyle="dashed",
    # )

    plt.xlabel("test log loss\n(baseline detector)")
    plt.ylabel("test log loss\n(calibrated detector)")

    custom_grid(plt.gca())
    plt.xlim(1e-1, 2)
    plt.ylim(1e-1, 2)
    plt.xscale("log")
    plt.yscale("log")

    plt.xticks(np.append(np.arange(0.1, 1, 0.1), 2), [], minor=True)
    plt.yticks(np.append(np.arange(0.1, 1, 0.1), 2), [], minor=True)
    plt.xticks([0.1, 1], [0.1, 1], minor=False)
    plt.yticks([0.1, 1], [0.1, 1], minor=False)
    # plt.grid()
    plt.scatter(
        [], [], color=mcolors.XKCD_COLORS["xkcd:neon pink"], label="clean cal.", s=12
    )
    plt.scatter(
        [], [], color=mcolors.XKCD_COLORS["xkcd:neon blue"], label="noisy cal.", s=12
    )
    # plt.scatter([], [], color="black", label="clean calib.", s=12)
    # plt.scatter([], [], color="black", label="noisy calib.", marker="x", s=12)
    plt.legend(edgecolor="black", handletextpad=0.4, handlelength=1, loc="upper left")
    # plt.title(f"{bottom['clean']}W-{top['clean']}L  {bottom['noisy']}W-{top['noisy']}L")

    tf.savefig(
        f"figures/perf_clean_vs_noisy_calib/{prefix.replace('/', '_')}_{exp_alt}_{k}"
    )
    plt.show()


if not os.path.exists("figures/perf_clean_vs_noisy_calib"):
    os.mkdir("figures/perf_clean_vs_noisy_calib")

plot_perf_clean_vs_noisy_calib(prefix="weak/klm", k="clean", exp_alt="allclass")
plot_perf_clean_vs_noisy_calib(prefix="weak/klm", k="oracl", exp_alt="allclass")

In [ ]:
import re

from sklearn.linear_model import LinearRegression


def plot_improv_ece_improv_detect(prefix, k, exp_alt):
    mapping = {"oracl": "test", "clean": "val", "noisy": "noisy_val"}
    best_model = (
        all_results[exp_alt][prefix]
        .groupby(["dataset_name", "detector_name"])[f"logl_{mapping[k]}"]
        .idxmin()
    )

    # results_cv = all_results_cv[exp_alt][prefix][metric]
    logl_cv = all_val_cv[exp_alt][prefix][metric]
    logl_cv_for_norm = all_val_cv["allclass"][prefix][metric]

    logl_cv_none = logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "none"].values
    # logl_cv_random = logl_cv[k].loc[logl_cv[k].index == "random"].values
    # logl_cv_wood = logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "wood"].values
    logl_cv_silver = (
        logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "silver"].values
    )
    # logl_cv_gold = logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "gold"].values

    # norm_low = (logl_cv_none + logl_cv_wood) / 2
    # norm_high = (logl_cv_silver + logl_cv_gold) / 2

    if False and exp_alt == "relabel":
        norm_low = logl_cv_random
    else:
        norm_low = logl_cv_none
    norm_high = logl_cv_silver

    print(k, norm_low.shape, norm_high.shape, logl_cv[k].shape)
    logl_cv_norm = (logl_cv[k] - norm_low) / (norm_high - norm_low + 1e-10)
    logl_cv_norm = (2 - logl_cv_norm) * 100

    ece = (
        all_results[exp_alt][prefix]
        .iloc[best_model]
        .pivot(
            index="detector_name",
            columns="dataset_name",
            values=f"ece_{mapping[k]}",
        )
    )

    logl = (
        all_results[exp_alt][prefix]
        .iloc[best_model]
        .pivot(
            index="detector_name",
            columns="dataset_name",
            values=f"logl_{mapping[k]}",
        )
    )

    tf.figure(width=TMLR_textwidth * (2 / 5), ratio=1, pad=0.5)

    X1, X2, Y1, Y2 = [], [], [], []
    for detector in set(detectors) - set(baselines + ["random"]):
        if len(detector.split("_")) > 2:
            continue

        print(detector)
        y1 = (
            ece.loc[ece.index == detector].values
            / ece.loc[ece.index == f"{detector}_isotonic_0.2"].values
        )

        x1 = (
            logl.loc[logl.index == detector].values
            / logl.loc[logl.index == f"{detector}_isotonic_0.2"].values
        )

        # y2 = (
        #     ece.loc[ece.index == detector].values
        #     / ece.loc[ece.index == f"{detector}_sigmoid_0.2"].values
        # )

        # x2 = (
        #     logl.loc[logl.index == detector].values
        #     / logl.loc[logl.index == f"{detector}_sigmoid_0.2"].values
        # )

        # plt.scatter(x1, y1, color=full_detector_colors[detector], s=12, alpha=0.7)
        plt.scatter(x1, y1, color=mcolors.XKCD_COLORS["xkcd:neon purple"], s=12, alpha=0.7)

        # plt.scatter(
        #     x2, y2, color=full_detector_colors[detector], s=12, alpha=0.7, marker="x"
        # )
        X1.append(x1)
        # X2.append(x2)
        Y1.append(y1)
        # Y2.append(y2)

    X1, Y1 = (
        np.concatenate(X1),
        # np.concatenate(X2),
        np.concatenate(Y1),
        # np.concatenate(Y2),
    )

    lr1 = LinearRegression(fit_intercept=True).fit(X1.reshape(-1, 1), Y1.ravel())
    # lr2 = LinearRegression(fit_intercept=True).fit(X2.reshape(-1, 1), Y2.ravel())

    # lr = LinearRegression(fit_intercept=True).fit(
    #     np.concatenate((X1, X2)).reshape(-1, 1), np.concatenate((Y1, Y1)).ravel()
    # )
    plt.plot(
        (xx := np.linspace(0.5, 2, num=200, endpoint=True)),
        lr1.predict(xx.reshape(-1, 1)),
        color="black",
    )
    # print(lr1.score(X1.reshape(-1, 1), Y1.ravel()), lr1.coef_, lr1.intercept_)
    # print(lr2.score(X2.reshape(-1, 1), Y2.ravel()), lr2.coef_, lr2.intercept_)

    # plt.plot(
    #     (xx := np.linspace(0.5, 2, num=200, endpoint=True)),
    #     lr1.predict(xx.reshape(-1, 1)),
    #     color=mcolors.XKCD_COLORS["xkcd:neon purple"],
    # )
    # plt.plot(
    #     (xx := np.linspace(0.5, 2, num=200, endpoint=True)),
    #     lr2.predict(xx.reshape(-1, 1)),
    #     color=mcolors.XKCD_COLORS["xkcd:neon green"],
    # )
    # plt.plot([0, 0.5], [0, 0.5], color="black")

    plt.ylabel("test ECE improvement")
    plt.xlabel("test log loss improvement")

    # plt.scatter([], [], color="black", label="isotonic", s=12)
    # plt.scatter([], [], color="black", label="sigmoid", s=12, marker="x")
    # plt.legend(loc="lower right", edgecolor="black", handletextpad=0.4, handlelength=1)

    plt.yscale("log")
    # plt.xscale("log")

    custom_grid(plt.gca())
    plt.ylim(0.1, 20)
    plt.xlim(0.5, 2)

    # plt.xticks(np.arange(0.1, 1, 0.1), [], minor=True)
    plt.yticks([0.1, 1, 10], [0.1, 1, 10], minor=False)

    plt.xticks(np.arange(0.5, 2, 0.1), [], minor=True)
    plt.xticks([0.5, 1, 2.0], [0.5, 1, 2.0], minor=False)


    tf.savefig(
        f"figures/improv_ece_improv_detect/{prefix.replace('/', '_')}_{exp_alt}_{k}"
    )
    plt.show()


if not os.path.exists("figures/improv_ece_improv_detect"):
    os.mkdir("figures/improv_ece_improv_detect")

plot_improv_ece_improv_detect(prefix="weak/klm", k="clean", exp_alt="allclass")
plot_improv_ece_improv_detect(prefix="weak/klm", k="oracl", exp_alt="allclass")

In [ ]:
from scipy.stats import spearmanr

default_calibration_size = 1.0


def plot_corr_sig_iso(prefix, k, exp_alt):
    mapping = {"oracl": "test", "clean": "val", "noisy": "noisy_val"}
    best_model = (
        all_results[exp_alt][prefix]
        .groupby(["dataset_name", "detector_name"])[f"logl_{mapping[k]}"]
        .idxmin()
    )

    logl = (
        all_results[exp_alt][prefix]
        .iloc[best_model]
        .pivot(
            index="detector_name",
            columns="dataset_name",
            values="logl_test",
        )
    )

    tf.figure(width=TMLR_textwidth * (2 / 5), ratio=1, pad=0.5)

    X, Y = [], []
    for detector in set(detectors) - set(baselines + ["random"]):
        if len(detector.split("_")) > 2:
            continue

        x = logl.loc[logl.index == f"{detector}_isotonic_{default_calibration_size}"].values
        y = logl.loc[logl.index == f"{detector}_sigmoid_{default_calibration_size}"].values

        plt.scatter(
            x, y, color=mcolors.XKCD_COLORS["xkcd:neon purple"], s=12, alpha=0.7
        )
        X.append(x.ravel())
        Y.append(y.ravel())

    X, Y = np.concatenate(X), np.concatenate(Y)

    plt.xlabel("test log loss\n(calibrated isotonic detector)")
    plt.ylabel("test log loss\n(calibrated sigmoid detector)")

    # print(X.shape)
    # plt.text(
    #     # -0.4,
    #     # -0.65,
    #     0.7,
    #     0.15,
    #     f"$r_s$ = {round(spearmanr(X, Y).statistic.item(), 2)}",
    #     fontdict={"font": "Times New Roman", "size": 9},
    #     bbox=dict(
    #         facecolor="none", edgecolor="black", linewidth=1, boxstyle="round,pad=0.2"
    #     ),
    #     math_fontfamily="cm"
    # )
    # plt.scatter([], [], color="black", label="isotonic.", s=12)
    # plt.scatter([], [], color="black", label="sigmoid", marker="x", s=12)

    plt.xscale("log")
    plt.yscale("log")

    plt.plot([0.1, 2], [0.1, 2], color="black")

    custom_grid(plt.gca())
    plt.ylim(0.1, 2)
    plt.xlim(0.1, 2)

    plt.xticks(np.arange(0.1, 1, 0.1), [], minor=True)
    plt.xticks([0.1, 1], [0.1, 1], minor=False)

    plt.yticks(np.arange(0.1, 1, 0.1), [], minor=True)
    plt.yticks([0.1, 1], [0.1, 1], minor=False)

    # plt.legend(loc="lower right", edgecolor="black", handletextpad=0.4, handlelength=1)

    tf.savefig(f"figures/plot_corr_sig_iso/{prefix.replace('/', '_')}_{exp_alt}_{k}")
    plt.show()


if not os.path.exists("figures/plot_corr_sig_iso"):
    os.mkdir("figures/plot_corr_sig_iso")

plot_corr_sig_iso(prefix="weak/klm", k="clean", exp_alt="allclass")
plot_corr_sig_iso(prefix="weak/klm", k="oracl", exp_alt="allclass")

In [ ]:

from sklearn.linear_model import HuberRegressor, LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures


def plot_improv_ece_improv_detect_2(prefix, k, exp_alt):
    mapping = {"oracl": "test", "clean": "val", "noisy": "noisy_val"}
    best_model = (
        all_results[exp_alt][prefix]
        .groupby(["dataset_name", "detector_name"])[f"logl_{mapping[k]}"]
        .idxmin()
    )

    # # results_cv = all_results_cv[exp_alt][prefix][metric]
    # logl_cv = all_val_cv[exp_alt][prefix][metric]
    # logl_cv_for_norm = all_val_cv["allclass"][prefix][metric]

    # logl_cv_none = logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "none"].values
    # # logl_cv_random = logl_cv[k].loc[logl_cv[k].index == "random"].values
    # # logl_cv_wood = logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "wood"].values
    # logl_cv_silver = (
    #     logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "silver"].values
    # )
    # # logl_cv_gold = logl_cv_for_norm[k].loc[logl_cv_for_norm[k].index == "gold"].values

    # # norm_low = (logl_cv_none + logl_cv_wood) / 2
    # # norm_high = (logl_cv_silver + logl_cv_gold) / 2

    # if False and exp_alt == "relabel":
    #     norm_low = logl_cv_random
    # else:
    #     norm_low = logl_cv_none
    # norm_high = logl_cv_silver

    # print(k, norm_low.shape, norm_high.shape, logl_cv[k].shape)
    # logl_cv_norm = (logl_cv[k] - norm_low) / (norm_high - norm_low + 1e-10)
    # logl_cv_norm = (2 - logl_cv_norm) * 100

    ece = (
        all_results[exp_alt][prefix]
        .iloc[best_model]
        .pivot(
            index="detector_name",
            columns="dataset_name",
            values="ece_train",
        )
    )
    print(ece)

    logl = (
        all_results[exp_alt][prefix]
        .iloc[best_model]
        .pivot(
            index="detector_name",
            columns="dataset_name",
            values=f"logl_{mapping[k]}",
        )
    )

    tf.figure(width=TMLR_textwidth * (2 / 5), ratio=1, pad=0.5)

    X1 = []
    Y1 = []
    X2 = []
    Y2 = []

    for detector in set(detectors) - set(baselines + ["random"]):
        if len(detector.split("_")) > 2:
            continue

        print(detector)
        x1 = (
            ece.loc[ece.index == "none"].values
            / ece.loc[ece.index == "isotonic"].values
        )

        y1 = (
            logl.loc[logl.index == detector].values
            / logl.loc[logl.index == f"{detector}_isotonic_0.2"].values
        )

        x2 = (
            ece.loc[ece.index == "none"].values / ece.loc[ece.index == "sigmoid"].values
        )

        y2 = (
            logl.loc[logl.index == detector].values
            / logl.loc[logl.index == f"{detector}_sigmoid_0.2"].values
        )

        plt.scatter(x1, y1, color=full_detector_colors[detector], s=12, alpha=0.7)
        plt.scatter(
            x2, y2, color=full_detector_colors[detector], s=12, alpha=0.7, marker="x"
        )
        X1.extend(x1.tolist())
        X2.extend(x2.tolist())
        Y1.extend(y1.tolist())
        Y2.extend(y2.tolist())

    # plt.plot([0, 0.5], [0, 0.5], color="black")

    lr1 = make_pipeline(PolynomialFeatures(), LinearRegression())
    lr2 =make_pipeline(PolynomialFeatures(), LinearRegression())
    lr1.fit(np.asarray(X1).reshape(-1, 1), np.asarray(Y1).ravel())
    lr2.fit(np.asarray(X2).reshape(-1, 1), np.asarray(Y2).ravel())

    print(np.linspace(*plt.xlim(), num=100))
    plt.plot(
        xx := np.linspace(*plt.xlim(), num=100),
        lr1.predict(xx.reshape(-1, 1)),
    )
    plt.plot(
        xx := np.linspace(*plt.xlim(), num=100),
        lr2.predict(xx.reshape(-1, 1)),
    )

    plt.xlabel("train ECE improvement")
    plt.ylabel("test log loss improvement")

    # plt.scatter([], [], color="black", label="isotonic.", s=12)
    # plt.scatter([], [], color="black", label="sigmoid", marker="x", s=12)

    # plt.xscale("log")
    # plt.yscale("log")

    custom_grid(plt.gca())
    # plt.ylim(0.1, 20)
    # plt.xlim(1, 100)

    # plt.xticks(np.arange(0.1, 1, 0.1), [], minor=True)
    # plt.xticks([0.1, 1, 10], [0.1, 1, 10], minor=False)

    # plt.yticks(np.arange(0.5, 2, 0.1), [], minor=True)
    # plt.yticks([0.5, 1, 2.0], [0.5, 1, 2.0], minor=False)

    # plt.legend(loc="lower right", edgecolor="black", handletextpad=0.4, handlelength=1)

    tf.savefig(
        f"figures/improv_ece_improv_detect2/{exp_alt}_{prefix.replace('/', '_')}"
    )
    plt.show()


if not os.path.exists("figures/improv_ece_improv_detect2"):
    os.mkdir("figures/improv_ece_improv_detect2")

plot_improv_ece_improv_detect_2(prefix="weak/klm", k="clean", exp_alt="allclass")

In [ ]:
def plot_calib_noisy_clean(prefix, k, exp_alt):

    mapping = {"oracl": "test", "clean": "val", "noisy": "noisy_val"}
    best_model = (
        all_results[exp_alt][prefix]
        .groupby(["dataset_name", "detector_name"])[f"logl_{mapping[k]}"]
        .idxmin()
    )

    ece = {}
    for eval_split in ["val", "test", "noisy_val"]:

        ece[eval_split] = (
            all_results[exp_alt][prefix]
            .iloc[best_model]
            .pivot(
                index="detector_name",
                columns="dataset_name",
                values=f"ece_{eval_split}",
            )
        )

    alternatives = ["adjusted", "calibrated"]

    for base in ["gb", "klm"]:

        for alternative in alternatives:
            tf.figure(width=TMLR_textwidth * (2 / 5), ratio=1, pad=0.5)

            for detector in set(detectors) - set(baselines + ["random"]):

                if not detector.startswith(base) or any(
                    detector.endswith(alternative) for alternative in alternatives
                ):
                    continue

                x1 = ece["noisy_val"].loc[ece["noisy_val"].index == detector]
                y1 = ece["val"].loc[ece["val"].index == detector]

                x2 = ece["noisy_val"].loc[
                    ece["noisy_val"].index == f"{detector}_{alternative}"
                ]
                y2 = ece["val"].loc[ece["val"].index == f"{detector}_{alternative}"]

                for i in range(len(x1.values.ravel())):
                    plt.annotate(
                        "",
                        (x2.values.ravel()[i], y2.values.ravel()[i]),
                        (x1.values.ravel()[i], y1.values.ravel()[i]),
                        arrowprops={
                            "arrowstyle": "->",
                            "color": colormaps["tab20"].colors[i],
                            "shrinkA": 0,
                            "shrinkB": 0,
                        },
                    )

            xlims, ylims = plt.xlim(), plt.ylim()
            xy_min = min(xlims[0], ylims[0])
            xy_max = max(xlims[1], ylims[1])

            plt.plot([0, 0.5], [0, 0.5], color="black")

            plt.xlabel("noisy ECE")
            plt.ylabel("clean ECE")

            custom_grid(plt.gca())
            plt.xlim(0, 0.5)
            plt.ylim(0, 0.5)

            tf.savefig(
                f"figures/calib_clean_vs_noisy/{exp_alt}_{prefix.replace('/','_')}_{alternative}_{base}"
            )
            plt.show()


if not os.path.exists("figures/calib_clean_vs_noisy"):
    os.mkdir("figures/calib_clean_vs_noisy")

plot_calib_noisy_clean(prefix="weak/klm", k="oracl", exp_alt="allclass")

In [ ]:
def plot_clean_noisy(prefix, exp_alt):
    logl_cv_noisy = all_val_cv[exp_alt][prefix]["logl"]["noisy"]
    logl_cv_clean = all_val_cv[exp_alt][prefix]["logl"]["clean"]
    logl_cv_noisy_none = logl_cv_noisy.loc[logl_cv_noisy.index == "none"]
    logl_cv_clean_none = logl_cv_clean.loc[logl_cv_clean.index == "none"]

    tf.figure(width=TMLR_textwidth * (2 / 5), ratio=1, pad=0.5)

    for detector in set(detectors) - set(baselines):

        perf_noisy = logl_cv_noisy.loc[logl_cv_noisy.index == detector]
        perf_clean = logl_cv_clean.loc[logl_cv_clean.index == detector]

        plt.scatter(
            logl_cv_noisy_none,
            perf_noisy,
            color=full_detector_colors[detector],
            s=12,
            alpha=0.7,
        )

        plt.scatter(
            logl_cv_clean_none,
            perf_clean,
            color=full_detector_colors[detector],
            s=12,
            alpha=0.7,
            marker="x",
        )

    xlims, ylims = plt.xlim(), plt.ylim()
    xy_min = min(xlims[0], ylims[0])
    xy_max = max(xlims[1], ylims[1])

    plt.plot([xy_min, xy_max], [xy_min, xy_max], color="black")

    plt.xlabel("test log loss\n(no filtering)")
    plt.ylabel("test log loss\n(detect + filter)")

    custom_grid(plt.gca())
    plt.xlim(1e-1, 1)
    plt.ylim(1e-1, 1)
    plt.xscale("log")
    plt.yscale("log")

    plt.xticks(np.arange(0.1, 1, 0.1), [], minor=True)
    plt.yticks(np.arange(0.1, 1, 0.1), [], minor=True)
    plt.xticks([0.1, 1], [0.1, 1], minor=False)
    plt.yticks([0.1, 1], [0.1, 1], minor=False)

    plt.scatter([], [], color="black", label=f"clean valid.", marker="x", s=12)
    plt.scatter([], [], color="black", label=f"noisy valid.", s=12)
    plt.legend()

    tf.savefig(f"figures/clean_vs_noisy/{exp_alt}_{prefix.replace('/','_')}")
    plt.show()


plot_clean_noisy(prefix="weak/klm", exp_alt="allclass")

In [ ]:
prefix = "weak/klm"
exp_alt = "allclass"

hp_pivot = all_results_cv[exp_alt][prefix]["logl"]["clean"].pivot(
    values="params_classifier", index="dataset_name", columns="detector_name"
)
res = all_results[exp_alt][prefix]
res_none = all_results_cv[exp_alt][prefix]["logl"]["clean"][
    all_results_cv[exp_alt][prefix]["logl"]["clean"]["detector_name"] == "none"
][["dataset_name", "logl_test"]].set_index("dataset_name")
res_silver = all_results_cv[exp_alt][prefix]["logl"]["clean"][
    all_results_cv[exp_alt][prefix]["logl"]["clean"]["detector_name"] == "silver"
][["dataset_name", "logl_test"]].set_index("dataset_name")
res_gold = all_results_cv[exp_alt][prefix]["logl"]["clean"][
    all_results_cv[exp_alt][prefix]["logl"]["clean"]["detector_name"] == "gold"
][["dataset_name", "logl_test"]].set_index("dataset_name")

mult = 1.5
fig, axes = plt.subplots(4, 5, figsize=(12 * mult, 8 * mult))

for i, dataset in enumerate(datasets):
    axis = axes[i // 5][i % 5]

    for detector in set(detectors) - {"gb_consensus", "klm_consensus"} - set(baselines):
        hp_this = hp_pivot.loc[dataset][detector]
        logl_this = res[(res.dataset_name == dataset) & (res.detector_name == detector)]
        logl_this = logl_this[logl_this.params_classifier.apply(lambda x: x == hp_this)]

        split_quantiles = []
        logls = []
        for p_split, logl in logl_this[["params_splitter", "logl_test"]].itertuples(
            index=False
        ):
            split_quantiles.append(p_split["quantile"] if "quantile" in p_split else p_split["threshold"])
            logls.append(logl)

        axis.plot(split_quantiles, logls, label=detector, color=full_detector_colors[detector])

    axis.plot([0, 1], [res_none.loc[dataset].values] * 2, color="red")
    axis.plot([0, 1], [res_silver.loc[dataset].values] * 2, color="silver")
    axis.plot([0, 1], [res_gold.loc[dataset].values] * 2, color="gold")

    custom_grid(axis)
    axis.set_title(dataset)
    # axis.legend()
    axis.set_xlabel("split quantile")
    axis.set_ylabel("log loss test")
    axis.set_xlim()
    axis.set_yscale("log")

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right')

fig.tight_layout()
plt.show()

In [ ]:
detect_dirs = [
    (
        "weak",
        os.path.join(os.path.expanduser("~"), f"{output_dir}/../ecai-sigmoid-2/weak"),
    ),
    # (
    #     "noise",
    #     os.path.join(os.path.expanduser("~"), f"{output_dir}/detect/noise"),
    # ),
]

exp_alt = "allclass"

all_detect = dict()
for prefix, dir in detect_dirs:
    results_ = []
    methods = os.listdir(dir)

    for method in methods:
        if not os.path.isdir(os.path.join(result_dir, method)):
            continue
        for fname in os.listdir(os.path.join(dir, method)):
            dataset, ext = fname.split(".")
            if ext != "json":
                continue

            with open(os.path.join(dir, method, f"{dataset}.json")) as f:
                results_.append(pd.read_json(f, orient="records"))

    results_ = pd.concat(results_)

    all_detect[prefix] = results_.reset_index(drop=True)
    all_detect[prefix]["detect_run"] = all_detect[prefix].groupby(["dataset_name", "detector_name"]).cumcount()+1

# for k, v in all_results[exp_alt].items():
#     v.loc[pd.isna(v["params_detector"]), "params_detector"] = v.loc[
#         pd.isna(v["params_detector"]), "params"
#     ]
#     v.drop("params", inplace=True, axis=1)

all_merged = dict()
# for prefix in ["weak", "noise"]:
for prefix in ["weak"]:
    all_results[exp_alt][prefix + "/klm"]["params_detector"] = all_results[exp_alt][prefix + "/klm"][
        "params_detector"
    ].astype(str)
    all_detect[prefix]["params"] = all_detect[prefix]["params"].astype(str)

    all_merged[prefix] = pd.merge(
        all_results[exp_alt][prefix + "/klm"],
        all_detect[prefix],
        left_on=["dataset_name", "detector_name", "params_detector"],
        right_on=["dataset_name", "detector_name", "params"],
    )

    all_merged[prefix]["params_detector"] = all_merged[prefix]["params_detector"].apply(
        eval
    )

In [ ]:
detect_path = os.path.join(os.path.expanduser("~"), f"{output_dir}/../ecai-sigmoid-2")
import json
import h5py
import numpy as np
import pandas as pd
from scipy.stats import entropy


class TrustScoreReader:

    def __init__(self, base_path, dataset, detector):

        with open(os.path.join(base_path, detector, f"{dataset}.json"), mode="r") as f:
            self.results_json = json.load(f)
        self.results_hdf5 = h5py.File(
            os.path.join(base_path, detector, f"{dataset}.hdf5"), "r"
        )

        assert len(self.results_hdf5["trust_scores"]) == len(self.results_json)

    def get(self, i):
        return self.results_json[i], self.results_hdf5[f"trust_scores/{i}"][...]

    def length(self):
        return len(self.results_json)

In [ ]:
from datasets import get_weak_datasets, datasets_ranked_by_time
import os

datasets_folder = os.path.join(os.path.expanduser("~"), "datasets")

datasets = list(
    filter(
        lambda dataset: dataset not in ["imdb136", "cifar10"], datasets_ranked_by_time
    )
)

weak_datasets = get_weak_datasets(
    cache_folder=datasets_folder,
    corruption="weak",
    seed=1,
    datasets=datasets,
)

# noise_datasets = get_weak_datasets(
#     cache_folder=datasets_folder,
#     corruption="noise",
#     seed=1,
#     datasets=datasets_ranked_by_time,
# )

loaded_datasets = dict(weak=dict(weak_datasets))

In [ ]:
from itertools import product
import matplotlib.colors as mcolors


def plot_class_balance(prefix, reference="clean", k="oracl", exp_alt="allclass"):
    mapping = {"oracl": "test", "clean": "val", "noisy": "noisy_val"}
    best_cv = (
        all_merged[prefix]
        .groupby(["dataset_name", "detector_name"])[f"logl_{mapping[k]}"]
        .idxmin()
    )
    best_trust_scores_index = (
        all_merged[prefix]
        .iloc[best_cv]
        .pivot(
            index="detector_name",
            columns="dataset_name",
            values="detect_run",
        )
    )
    best_split = (
        all_merged[prefix]
        .iloc[best_cv]
        .pivot(
            index="detector_name",
            columns="dataset_name",
            values="params_splitter",
        )
        .map(
            lambda d: d["quantile"] if "quantile" in d.keys() else d["threshold"],
            na_action="ignore",
        )
    )

    # best_threshold = (
    #     all_merged[prefix]
    #     .iloc[best_cv]
    #     .pivot(
    #         index="detector_name",
    #         columns="dataset_name",
    #         values="params_splitter",
    #     )
    #     .map(lambda dict: dict.get("threshold"))
    # )
    tf.figure(width=TMLR_textwidth * (2 / 5), ratio=1, pad=0.5)

    top = {detector_name: 0 for detector_name in best_trust_scores_index.index}
    bottom = {detector_name: 0 for detector_name in best_trust_scores_index.index}

    for detector_name, dataset_name in product(
        best_trust_scores_index.index, best_trust_scores_index.columns
    ):
        if (
            "gb" in detector_name
            or "nois" in detector_name
            or "forg" in detector_name
            or "sig" in detector_name
        ):
            continue

        if "iso" in detector_name and "0.2" not in detector_name:
            continue

        trust_score_reader = TrustScoreReader(
            os.path.join(detect_path, prefix), dataset_name, detector_name
        )
        idx = (best_trust_scores_index.loc[detector_name, dataset_name] - 1).item()
        if math.isnan(idx):
            print(detector_name, dataset_name)
            continue
        trust_scores = trust_score_reader.get(int(idx))[1]
        dataset = loaded_datasets[prefix][dataset_name]
        y_train_clean = dataset["train"]["target"]
        y_train_noisy = dataset["train"]["noisy_target"]
        y_test = dataset["test"]["target"]
        unlabeled = y_train_noisy == -1
        y_train_clean = np.asarray(y_train_clean)[~unlabeled]
        y_train_noisy = np.asarray(y_train_noisy)[~unlabeled]
        n_classes = len(np.unique(y_test))
        if "consensus" in detector_name:
            trusted = trust_scores >= best_split.loc[detector_name, dataset_name]
        else:
            trusted = trust_scores >= np.quantile(
                trust_scores, q=best_split.loc[detector_name, dataset_name]
            )
        prior_trusted = np.bincount(y_train_noisy[trusted], minlength=n_classes) / len(
            y_train_noisy[trusted]
        )
        prior_untrusted = np.bincount(
            y_train_noisy[~trusted], minlength=n_classes
        ) / len(y_train_noisy[~trusted])
        prior_clean = np.bincount(y_test, minlength=n_classes) / len(y_test)
        prior_noisy = np.bincount(y_train_noisy, minlength=n_classes) / len(
            y_train_noisy
        )
        x_clean = np.min(prior_clean) / np.max(prior_clean)
        x_noisy = np.min(prior_noisy) / np.max(prior_noisy)
        x = x_clean if reference == "clean" else x_noisy
        y = np.min(prior_trusted) / np.max(prior_trusted)
        plt.scatter(
            x,
            y,
            # color="tab:blue" if x_noisy > x_clean else "tab:orange",
            # color=full_detector_colors[detector_name],
            color="tab:orange"
            if "adjust" in detector_name
            else ("tab:green" if "isotonic" in detector_name else "tab:blue"),
            # marker=full_detector_markers[detector_name],
            # marker=(
            #     "x"
            #     if "adjust" in detector_name
            #     else ("o" if "calib" in detector_name else "s")
            # ),
            alpha=0.7,
            s=12,
        )
        top[detector_name] += np.sum(y > x)
        bottom[detector_name] += np.sum(y <= x)

    print({k: v for k, v in top.items() if "iso" in k and "0.2" in k and "nois" not in k}, {k: v for k, v in bottom.items() if "iso" in k and "0.2" in k and "nois" not in k})

    plt.ylabel("filtered class balance")
    if reference == "clean":
        plt.xlabel("clean class balance")
    else:
        plt.xlabel("noisy class balance")

    # plt.title(f"{top}/{bottom}")

    custom_grid(plt.gca())

    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.plot(np.linspace(0, 1, 1000), np.linspace(0, 1, 1000), color="black")

    plt.xticks(np.arange(0, 1, 0.1), [], minor=True)
    plt.yticks(np.arange(0, 1, 0.1), [], minor=True)
    plt.xticks([0, 1], [0, 1], minor=False)
    plt.yticks([0, 1], [0, 1], minor=False)

    if reference == "clean":
        plt.scatter([], [], color="tab:blue", label="base.", s=12)
        plt.scatter([], [], color="tab:orange", label="adj.", s=12)
        plt.scatter([], [], color="tab:green", label="cal.", s=12)
        plt.legend(edgecolor="black", handletextpad=0.4, handlelength=1)

    tf.savefig(f"figures/class_balance/{prefix.replace('/', '_')}_{reference}_{k}")
    plt.show()


if not os.path.exists("figures/class_balance"):
    os.mkdir("figures/class_balance")
# plot_class_balance("noise", reference="clean")
# plot_class_balance("noise", reference="noisy")
plot_class_balance("weak", reference="clean", k="clean")
plot_class_balance("weak", reference="noisy", k="clean")
plot_class_balance("weak", reference="clean", k="oracl")
plot_class_balance("weak", reference="noisy", k="oracl")

In [ ]:
from itertools import product
import matplotlib.colors as mcolors


sns.color_palette("Set2", len(detector_base))


def plot_class_balance(prefix, k="oracl", exp_alt="allclass"):
    mapping = {"oracl": "test", "clean": "val", "noisy": "noisy_val"}
    best_cv = (
        all_merged[prefix]
        .groupby(["dataset_name", "detector_name"])[f"logl_{mapping[k]}"]
        .idxmin()
    )
    best_trust_scores_index = (
        all_merged[prefix]
        .iloc[best_cv]
        .pivot(
            index="detector_name",
            columns="dataset_name",
            values="detect_run",
        )
    )
    best_split = (
        all_merged[prefix]
        .iloc[best_cv]
        .pivot(
            index="detector_name",
            columns="dataset_name",
            values="params_splitter",
        )
        .map(
            lambda d: d["quantile"] if "quantile" in d.keys() else d["threshold"],
            na_action="ignore",
        )
    )

    # best_threshold = (
    #     all_merged[prefix]
    #     .iloc[best_cv]
    #     .pivot(
    #         index="detector_name",
    #         columns="dataset_name",
    #         values="params_splitter",
    #     )
    #     .map(lambda dict: dict.get("threshold"))
    # )
    fig, ax1 = tf.subplots(width=TMLR_textwidth * (2 / 5), ratio=1)

    top = {detector_name: 0 for detector_name in best_trust_scores_index.index}
    bottom = {detector_name: 0 for detector_name in best_trust_scores_index.index}

    ax2 = ax1.twinx()

    for detector_name, dataset_name in product(
        best_trust_scores_index.index, best_trust_scores_index.columns
    ):
        if (
            "gb" in detector_name
            or "nois" in detector_name
            or "forg" in detector_name
            or "sig" in detector_name
        ):
            continue

        if "iso" in detector_name and "0.2" not in detector_name:
            continue

        trust_score_reader = TrustScoreReader(
            os.path.join(detect_path, prefix), dataset_name, detector_name
        )
        idx = (best_trust_scores_index.loc[detector_name, dataset_name] - 1).item()
        if math.isnan(idx):
            print(detector_name, dataset_name)
            continue
        trust_scores = trust_score_reader.get(int(idx))[1]
        dataset = loaded_datasets[prefix][dataset_name]
        y_train_clean = dataset["train"]["target"]
        y_train_noisy = dataset["train"]["noisy_target"]
        y_test = dataset["test"]["target"]
        unlabeled = y_train_noisy == -1
        y_train_clean = np.asarray(y_train_clean)[~unlabeled]
        y_train_noisy = np.asarray(y_train_noisy)[~unlabeled]
        n_classes = len(np.unique(y_test))
        if "consensus" in detector_name:
            trusted = trust_scores >= best_split.loc[detector_name, dataset_name]
        else:
            trusted = trust_scores >= np.quantile(
                trust_scores, q=best_split.loc[detector_name, dataset_name]
            )
        prior_trusted = np.bincount(y_train_noisy[trusted], minlength=n_classes) / len(
            y_train_noisy[trusted]
        )
        prior_untrusted = np.bincount(
            y_train_noisy[~trusted], minlength=n_classes
        ) / len(y_train_noisy[~trusted])
        prior_clean = np.bincount(y_test, minlength=n_classes) / len(y_test)
        prior_noisy = np.bincount(y_train_noisy, minlength=n_classes) / len(
            y_train_noisy
        )
        clean = np.min(prior_clean) / np.max(prior_clean)
        noisy = np.min(prior_noisy) / np.max(prior_noisy)
        filtered = np.min(prior_trusted) / np.max(prior_trusted)
        if not "isotonic" in detector_name:
            continue
        ax1.scatter(
            clean,
            filtered,
            # color="tab:blue" if x_noisy > x_clean else "tab:orange",
            # color=full_detector_colors[detector_name],
            # color="tab:orange"
            # if "adjust" in detector_name
            # else ("tab:green" if "isotonic" in detector_name else "tab:blue"),
            color=mcolors.XKCD_COLORS["xkcd:neon pink"],
            # marker=full_detector_markers[detector_name],
            # marker=(
            #     "x"
            #     if "adjust" in detector_name
            #     else ("o" if "calib" in detector_name else "s")
            # ),
            marker="o"
            if "isotonic" in detector_name
            else ("x" if "adj" in detector_name else "s"),
            alpha=0.7,
            s=12,
        )
        ax2.scatter(
            clean,
            noisy,
            # color="tab:blue" if x_noisy > x_clean else "tab:orange",
            # color=full_detector_colors[detector_name],
            # color="tab:orange"
            # if "adjust" in detector_name
            # else ("tab:green" if "isotonic" in detector_name else "tab:blue"),
            color=mcolors.XKCD_COLORS["xkcd:neon blue"],
            # marker=full_detector_markers[detector_name],
            # marker=(
            #     "x"
            #     if "adjust" in detector_name
            #     else ("o" if "calib" in detector_name else "s")
            # ),
            alpha=0.7,
            s=12,
            marker="o"
            if "isotonic" in detector_name
            else ("x" if "adj" in detector_name else "s"),
        )

    ax1.set_ylabel("clean class balance")
    # fig.gca().spines["left"].set_color(mcolors.XKCD_COLORS["xkcd:neon blue"])
    ax2.set_ylabel("noisy class balance")
    # top[detector_name] += np.sum(y > x)
    # bottom[detector_name] += np.sum(y <= x)

    ax1.set_xlabel("filtered class balance")

    # plt.title(f"{top}/{bottom}")

    custom_grid(plt.gca())

    plt.xlim(0, 1)
    ax1.set_ylim(0, 1)
    ax2.set_ylim(0, 1)
    plt.plot(np.linspace(0, 1, 1000), np.linspace(0, 1, 1000), color="black")

    plt.xticks(np.arange(0, 1, 0.1), [], minor=True)
    plt.xticks([0, 1], [0, 1], minor=False)
    ax1.set_yticks([0, 1], [0, 1], minor=False)
    ax1.set_yticks(np.arange(0, 1, 0.1), [], minor=True)
    ax2.set_yticks([0, 1], [0, 1], minor=False)
    ax2.set_yticks(np.arange(0, 1, 0.1), [], minor=True)

    # plt.scatter([], [], color="tab:blue", label="base.", s=12)
    # plt.scatter([], [], color="tab:orange", label="adj.", s=12)
    # plt.scatter([], [], color="tab:green", label="cal.", s=12)
    plt.scatter([], [], color=mcolors.XKCD_COLORS["xkcd:neon pink"], label="clean", s=12)
    plt.scatter([], [], color=mcolors.XKCD_COLORS["xkcd:neon blue"], label="noisy", s=12)
    plt.legend(edgecolor="black", handletextpad=0.4, handlelength=1)

    # plt.scatter([], [], color="black", marker="o", label="cal.", s=12)
    # plt.scatter([], [], color="black", marker="x", label="adj.", s=12)
    # plt.scatter([], [], color="black", marker="s", label="base", s=12)
    # plt.legend(edgecolor="black", handletextpad=0.4, handlelength=1)

    tf.savefig(f"figures/class_balance_2/{prefix.replace('/', '_')}_{k}")
    plt.show()


if not os.path.exists("figures/class_balance_2"):
    os.mkdir("figures/class_balance_2")
# plot_class_balance("noise", reference="clean")
# plot_class_balance("noise", reference="noisy")
plot_class_balance("weak", k="clean")
plot_class_balance("weak", k="oracl")
# plot_class_balance("weak", reference="noisy", k="oracl")

In [ ]:
from scipy.stats import expectile
import matplotlib.colors as mcolors

sns.color_palette("Set2", len(detector_base))


def plot_minority_removed(
    prefix, k="oracl", exp_alt="allclass", label="clean", with_adjust=False
):
    mapping = {"oracl": "test", "clean": "val", "noisy": "noisy_val"}
    best_cv = (
        all_merged[prefix]
        .groupby(["dataset_name", "detector_name"])[f"logl_{mapping[k]}"]
        .idxmin()
    )
    best_trust_scores_index = (
        all_merged[prefix]
        .iloc[best_cv]
        .pivot(
            index="detector_name",
            columns="dataset_name",
            values="detect_run",
        )
    )

    # best_threshold = (
    #     all_merged[prefix]
    #     .iloc[best_cv]
    #     .pivot(
    #         index="detector_name",
    #         columns="dataset_name",
    #         values="params_splitter",
    #     )
    #     .map(lambda dict: dict.get("threshold"))
    # )
    fig, ax = tf.subplots(width=TMLR_textwidth * (2 / 5), ratio=1)

    results_per_method = {"none": [], "adjusted": [], "isotonic": [], "sigmoid": []}
    cnt_per_method = {"isotonic": 0, "sigmoid": 0, "none": 0, "adjusted": 0}
    splits_per_method = {
        "isotonic": None,
        "sigmoid": None,
        "none": None,
        "adjusted": None,
    }
    colors_per_method = {
        "isotonic": mcolors.XKCD_COLORS["xkcd:neon blue"],
        "sigmoid": mcolors.XKCD_COLORS["xkcd:neon pink"],
        "none": mcolors.XKCD_COLORS["xkcd:neon green"],
        "adjusted": mcolors.XKCD_COLORS["xkcd:neon purple"],
    }
    resolution = 100

    for detector_name, dataset_name in product(
        best_trust_scores_index.index, best_trust_scores_index.columns
    ):
        if "gb" in detector_name or "nois" in detector_name or "forg" in detector_name:
            continue

        if "consensus" in detector_name:
            continue

        if "sig" in detector_name and "1.0" in detector_name:
            calibration_method = "sigmoid"
            if with_adjust:
                continue
        elif "iso" in detector_name and "1.0" in detector_name:
            calibration_method = "isotonic"
            if with_adjust:
                continue
        elif "adj" in detector_name:
            calibration_method = "adjusted"
            if not with_adjust:
                continue
        elif len(detector_name.split("_")) == 2:
            calibration_method = "none"
        else:
            continue

        trust_score_reader = TrustScoreReader(
            os.path.join(detect_path, prefix), dataset_name, detector_name
        )
        idx = (best_trust_scores_index.loc[detector_name, dataset_name] - 1).item()
        if math.isnan(idx):
            print(detector_name, dataset_name)
            continue
        trust_scores = trust_score_reader.get(int(idx))[1]
        dataset = loaded_datasets[prefix][dataset_name]
        y_train_clean = dataset["train"]["target"]
        y_train_noisy = dataset["train"]["noisy_target"]
        y_test = dataset["test"]["target"]
        unlabeled = y_train_noisy == -1
        y_train_clean = np.asarray(y_train_clean)[~unlabeled]
        y_train_noisy = np.asarray(y_train_noisy)[~unlabeled]
        if label == "clean":
            y_train = y_train_clean
        else:
            y_train = y_train_noisy
        n_classes = len(np.unique(y_test))
        if "consensus" in detector_name:
            splits = np.sort(np.unique(trust_scores))
        else:
            splits = np.quantile(trust_scores, q=np.linspace(0, 1, resolution))
        if splits_per_method[calibration_method] is None:
            splits_per_method[calibration_method] = splits
        trusted = trust_scores[None, :] >= splits[:, None]
        trusted[-1, :] = False
        if n_classes == 2:
            minority = y_train == np.argmin(
                np.bincount(y_train_noisy, minlength=n_classes)
            )
        else:
            priors = np.bincount(y_train, minlength=n_classes) / y_train.shape[0]
            minority_classes = np.argwhere(priors <= 1 / n_classes)
            minority = np.isin(y_train, minority_classes)

        trusted_minority = trusted * minority[None, :]
        minority_removed = 1 - trusted_minority.sum(axis=1) / minority.sum()
        # removed = 1 - trusted.sum(axis=1) / trusted.shape[1]
        # ax.plot(
        #     removed,
        #     minority_removed,
        #     color=colors_per_method[calibration_method],
        #     alpha=0.1,
        #     linewidth=0.5,
        # )

        cnt_per_method[calibration_method] += 1
        results_per_method[calibration_method].append(minority_removed)

    for calibration_method in ["none", "adjusted", "sigmoid", "isotonic"]:
        minority_removed = results_per_method[calibration_method]
        if cnt_per_method[calibration_method] == 0:
            continue
        ax.fill_between(
            np.linspace(0, 1, resolution),
            np.quantile(minority_removed, 0.25, axis=0),
            np.quantile(minority_removed, 0.75, axis=0),
            color=colors_per_method[calibration_method],
            # label=calibration_method,
            alpha=0.2,
        )
        ax.plot(
            np.linspace(0, 1, resolution),
            np.quantile(minority_removed, 0.5, axis=0),
            color=colors_per_method[calibration_method],
            label=calibration_method,
        )

    custom_grid(plt.gca())

    # plt.xlim(0, 1)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    custom_grid(plt.gca())

    # plt.plot(np.linspace(0, 1, 1000), np.linspace(0, 1, 1000), color="black")
    ax.axline((0, 0), slope=1, color="black", linewidth=0.8, linestyle="--")

    ax.set_xticks(np.arange(0, 1, 0.1), [], minor=True)
    ax.set_xticks([0, 1], [0, 1], minor=False)
    ax.set_yticks([0, 1], [0, 1], minor=False)
    ax.set_yticks(np.arange(0, 1, 0.1), [], minor=True)

    ax.set_ylabel("% removed minority examples")
    ax.set_xlabel("% removed examples")

    legend = plt.legend(
        edgecolor="black", handletextpad=0.4, handlelength=1, fancybox=False
    )
    legend.get_frame().set_linewidth(0.8)

    tf.savefig(
        f"figures/minority_removed/{prefix.replace('/', '_')}_{k}_{label}_{with_adjust}"
    )
    plt.show()


if not os.path.exists("figures/minority_removed"):
    os.mkdir("figures/minority_removed")
# plot_class_balance("noise", reference="clean")
# plot_class_balance("noise", reference="noisy")

for prefix, k, label, with_adjust in product(
    ["weak"], ["clean", "oracl"], ["clean", "noisy"], [True, False]
):
    plot_minority_removed(prefix, k=k, label=label, with_adjust=with_adjust)
# plot_minority_removed("weak", k="clean")
# plot_minority_removed("weak", k="oracl")
# plot_minority_removed("weak", k="clean", label="noisy")
# plot_minority_removed("weak", k="oracl", label="noisy")
